> **Exploratory notebook — not required for final reproduction.**
> 
> This notebook was used during analysis development to inspect the spatial overlap
> between GHS Urban Centre Database (UCDB) 2019 polygons and City Segments prediction
> GeoPackages. It is retained for transparency but does **not** need to be run to
> reproduce any manuscript output. All final outputs are generated by notebooks
> `01`–`04` in this folder.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

# ============================================================
# PATH CONFIGURATION  (portable — no hard-coded local paths)
# ============================================================
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "citysegmentdeprivation" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_EXTERNAL              = REPO_ROOT / "data_external"
ZENODO_DATA                = DATA_EXTERNAL / "zenodo"
UCDB_DIR                   = DATA_EXTERNAL / "ucdb"
GHSPOP_DIR                 = DATA_EXTERNAL / "ghspop"

OUTPUT_TABLES              = REPO_ROOT / "outputs" / "tables" / "revision2"
OUTPUT_TABLES_INTERMEDIATE = OUTPUT_TABLES / "intermediate"
OUTPUT_FIGURES             = REPO_ROOT / "outputs" / "figures" / "revision2"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

# ============================================================
# USER INPUTS
# ============================================================
# Derived UCDB + GHS-POP 2025 GeoPackage (output of notebook 01, archived on Zenodo)
UCDB_GPKG      = ZENODO_DATA / "GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2_with_GHSPOP2023.gpkg"
# City Segments prediction GeoPackages from Zenodo (DOI: 10.5281/zenodo.18788260)
CITYSEG_PARENT = ZENODO_DATA / "predictions"

UCDB_ID_COL      = "ID_HDC_G0"
UCDB_REGION_COL  = "GRGN_L1"
UCDB_COUNTRY_COL = "CTR_MN_NM"

CITYSEG_ID_COL      = "ID_HDC_G0"
CITYSEG_REGION_COL  = "REG1_GHSL"
CITYSEG_COUNTRY_COL = "CTR_MN_NM"

In [2]:
# Identify layer
ucdb_layer = gpd.list_layers(UCDB_GPKG).iloc[0]["name"]

ucdb = gpd.read_file(
    UCDB_GPKG,
    layer=ucdb_layer,
    columns=[UCDB_ID_COL, UCDB_REGION_COL, UCDB_COUNTRY_COL]
)

# Clean types
ucdb[UCDB_ID_COL] = ucdb[UCDB_ID_COL].astype(str)

# Keep unique cities
ucdb_unique = (
    ucdb[[UCDB_ID_COL, UCDB_REGION_COL, UCDB_COUNTRY_COL]]
    .dropna(subset=[UCDB_ID_COL])
    .drop_duplicates(subset=[UCDB_ID_COL])
)

# Region × Country breakdown
ucdb_region_country = (
    ucdb_unique
    .groupby([UCDB_REGION_COL, UCDB_COUNTRY_COL])[UCDB_ID_COL]
    .nunique()
    .reset_index(name="UCDB_Cities")
    .sort_values(["GRGN_L1", "UCDB_Cities"], ascending=[True, False])
)

# Region totals
ucdb_region_totals = (
    ucdb_unique
    .groupby(UCDB_REGION_COL)[UCDB_ID_COL]
    .nunique()
    .reset_index(name="UCDB_Cities")
)

In [3]:
cityseg_gpkgs = sorted(CITYSEG_PARENT.rglob("*.gpkg"))

records = []

for fp in cityseg_gpkgs:
    try:
        layer = gpd.list_layers(fp).iloc[0]["name"]

        gdf = gpd.read_file(
            fp,
            layer=layer,
            columns=[CITYSEG_ID_COL, CITYSEG_REGION_COL, CITYSEG_COUNTRY_COL]
        )

        gdf[CITYSEG_ID_COL] = gdf[CITYSEG_ID_COL].astype(str)

        subset = (
            gdf[[CITYSEG_ID_COL, CITYSEG_REGION_COL, CITYSEG_COUNTRY_COL]]
            .dropna(subset=[CITYSEG_ID_COL])
            .drop_duplicates(subset=[CITYSEG_ID_COL])
        )

        records.append(subset)

    except Exception as e:
        print(f"⚠️ Error reading {fp.name}: {e}")

cityseg_all = pd.concat(records, ignore_index=True)

cityseg_unique = cityseg_all.drop_duplicates(subset=[CITYSEG_ID_COL])

# Region × Country breakdown
cityseg_region_country = (
    cityseg_unique
    .groupby([CITYSEG_REGION_COL, CITYSEG_COUNTRY_COL])[CITYSEG_ID_COL]
    .nunique()
    .reset_index(name="CitySegments_Cities")
    .sort_values(["REG1_GHSL", "CitySegments_Cities"], ascending=[True, False])
)

# Region totals
cityseg_region_totals = (
    cityseg_unique
    .groupby(CITYSEG_REGION_COL)[CITYSEG_ID_COL]
    .nunique()
    .reset_index(name="CitySegments_Cities")
)

In [4]:
# --------------------------------------------
# Country counts per region (UCDB)
# --------------------------------------------
ucdb_region_country_counts = (
    ucdb_unique
    .groupby(UCDB_REGION_COL)[UCDB_COUNTRY_COL]
    .nunique()
    .reset_index(name="UCDB_Countries")
)

# --------------------------------------------
# Country counts per region (City Segments)
# --------------------------------------------
cityseg_region_country_counts = (
    cityseg_unique
    .groupby(CITYSEG_REGION_COL)[CITYSEG_COUNTRY_COL]
    .nunique()
    .reset_index(name="CitySegments_Countries")
)

In [5]:
# -------------------------------------------------------
# Merge city totals
# -------------------------------------------------------
region_comparison = pd.merge(
    ucdb_region_totals,
    cityseg_region_totals,
    left_on=UCDB_REGION_COL,
    right_on=CITYSEG_REGION_COL,
    how="outer"
)

region_comparison = region_comparison.drop(columns=[CITYSEG_REGION_COL])

# -------------------------------------------------------
# Merge country counts
# -------------------------------------------------------
region_comparison = pd.merge(
    region_comparison,
    ucdb_region_country_counts,
    on=UCDB_REGION_COL,
    how="left"
)

region_comparison = pd.merge(
    region_comparison,
    cityseg_region_country_counts,
    left_on=UCDB_REGION_COL,
    right_on=CITYSEG_REGION_COL,
    how="left"
)

region_comparison = region_comparison.drop(columns=[CITYSEG_REGION_COL])

# -------------------------------------------------------
# Fill missing
# -------------------------------------------------------
region_comparison = region_comparison.fillna(0)

# -------------------------------------------------------
# Coverage %
# -------------------------------------------------------
region_comparison["Coverage_%"] = (
    region_comparison["CitySegments_Cities"] /
    region_comparison["UCDB_Cities"].replace(0, pd.NA)
) * 100

# Optional: reorder columns nicely
region_comparison = region_comparison[
    [
        UCDB_REGION_COL,
        "UCDB_Countries",
        "CitySegments_Countries",
        "UCDB_Cities",
        "CitySegments_Cities",
        "Coverage_%"
    ]
].sort_values("UCDB_Cities", ascending=False)

region_comparison

,GRGN_L1,UCDB_Countries,CitySegments_Countries,UCDB_Cities,CitySegments_Cities,Coverage_%
1,Asia,49,30.0,7737,2751.0,35.556417
0,Africa,55,52.0,2805,1500.0,53.475936
3,Latin America and the Caribbean,30,21.0,1076,953.0,88.568773
2,Europe,41,1.0,1059,5.0,0.472144
4,Northern America,2,0.0,372,0.0,0.000000
5,Oceania,7,3.0,86,5.0,5.813953


In [6]:
country_comparison = pd.merge(
    ucdb_region_country,
    cityseg_region_country,
    left_on=[UCDB_REGION_COL, UCDB_COUNTRY_COL],
    right_on=[CITYSEG_REGION_COL, CITYSEG_COUNTRY_COL],
    how="outer"
)

# Clean duplicated region columns
country_comparison = country_comparison.drop(columns=[CITYSEG_REGION_COL])

country_comparison = country_comparison.rename(
    columns={CITYSEG_COUNTRY_COL: UCDB_COUNTRY_COL}
)

country_comparison = country_comparison.fillna(0)

country_comparison["Coverage_%"] = (
    country_comparison["CitySegments_Cities"] /
    country_comparison["UCDB_Cities"].replace(0, pd.NA)
) * 100

country_comparison = country_comparison.sort_values(
    ["GRGN_L1", "UCDB_Cities"],
    ascending=[True, False]
)

country_comparison

,GRGN_L1,CTR_MN_NM,UCDB_Cities,CitySegments_Cities,Coverage_%
17,Africa,Ethiopia,557,45.0,8.078995
37,Africa,Nigeria,483,318.0,65.838509
14,Africa,Egypt,190,181.0,95.263158
12,Africa,Democratic Republic of the Congo,160,101.0,63.125000
45,Africa,Sudan,121,48.0,39.669421
...,...,...,...,...,...
181,Oceania,New Zealand,8,0.0,0.000000
178,Oceania,Fiji,1,1.0,100.000000
179,Oceania,French Polynesia,1,0.0,0.000000
180,Oceania,New Caledonia,1,0.0,0.000000


In [7]:
def classify_city_size(pop):
    if pop < 500_000:
        return "Small (<500k)"
    elif pop < 1_000_000:
        return "Medium (500k–1M)"
    elif pop < 5_000_000:
        return "Large (1–5M)"
    elif pop < 10_000_000:
        return "Very large (5–10M)"
    else:
        return "Megacity (>10M)"

In [8]:
UCDB_POP_COL = "GHSPOP2023"

ucdb = gpd.read_file(
    UCDB_GPKG,
    layer=ucdb_layer,
    columns=[UCDB_ID_COL, UCDB_REGION_COL, UCDB_COUNTRY_COL, UCDB_POP_COL]
)

ucdb[UCDB_ID_COL] = ucdb[UCDB_ID_COL].astype(str)

ucdb_unique = (
    ucdb[[UCDB_ID_COL, UCDB_REGION_COL, UCDB_COUNTRY_COL, UCDB_POP_COL]]
    .dropna(subset=[UCDB_ID_COL])
    .drop_duplicates(subset=[UCDB_ID_COL])
)

ucdb_unique["CitySizeClass"] = ucdb_unique[UCDB_POP_COL].apply(classify_city_size)

In [9]:
CITYSEG_POP_COL = "POP_SEG"

records = []

for fp in cityseg_gpkgs:
    try:
        layer = gpd.list_layers(fp).iloc[0]["name"]

        gdf = gpd.read_file(
            fp,
            layer=layer,
            columns=[
                CITYSEG_ID_COL,
                CITYSEG_REGION_COL,
                CITYSEG_COUNTRY_COL,
                CITYSEG_POP_COL
            ]
        )

        gdf[CITYSEG_ID_COL] = gdf[CITYSEG_ID_COL].astype(str)

        records.append(gdf)

    except Exception as e:
        print(f"⚠️ Error reading {fp.name}: {e}")

cityseg_all = pd.concat(records, ignore_index=True)

# Aggregate population per city
cityseg_city_pop = (
    cityseg_all
    .groupby([CITYSEG_ID_COL, CITYSEG_REGION_COL, CITYSEG_COUNTRY_COL])[CITYSEG_POP_COL]
    .sum()
    .reset_index()
)

cityseg_city_pop["CitySizeClass"] = cityseg_city_pop[CITYSEG_POP_COL].apply(classify_city_size)

In [10]:
ucdb_region_size = (
    ucdb_unique
    .groupby([UCDB_REGION_COL, "CitySizeClass"])[UCDB_ID_COL]
    .nunique()
    .reset_index(name="UCDB_Cities")
)

In [11]:
cityseg_region_size = (
    cityseg_city_pop
    .groupby([CITYSEG_REGION_COL, "CitySizeClass"])[CITYSEG_ID_COL]
    .nunique()
    .reset_index(name="CitySegments_Cities")
)

In [12]:
region_size_comparison = pd.merge(
    ucdb_region_size,
    cityseg_region_size,
    left_on=[UCDB_REGION_COL, "CitySizeClass"],
    right_on=[CITYSEG_REGION_COL, "CitySizeClass"],
    how="outer"
)

region_size_comparison = region_size_comparison.drop(columns=[CITYSEG_REGION_COL])
region_size_comparison = region_size_comparison.fillna(0)

region_size_comparison["Coverage_%"] = (
    region_size_comparison["CitySegments_Cities"] /
    region_size_comparison["UCDB_Cities"].replace(0, pd.NA)
) * 100

region_size_comparison = region_size_comparison.sort_values(
    [UCDB_REGION_COL, "CitySizeClass"]
)

region_size_comparison

,GRGN_L1,CitySizeClass,UCDB_Cities,CitySegments_Cities,Coverage_%
0,Africa,Large (1–5M),69,105.0,152.173913
1,Africa,Medium (500k–1M),78,151.0,193.589744
2,Africa,Megacity (>10M),4,13.0,325.000000
3,Africa,Small (<500k),2644,1212.0,45.839637
4,Africa,Very large (5–10M),10,19.0,190.000000
5,Asia,Large (1–5M),230,131.0,56.956522
6,Asia,Medium (500k–1M),311,146.0,46.945338
7,Asia,Megacity (>10M),22,13.0,59.090909
8,Asia,Small (<500k),7145,2446.0,34.233730
9,Asia,Very large (5–10M),29,15.0,51.724138


In [13]:
ucdb_size_global = (
    ucdb_unique
    .groupby("CitySizeClass")[UCDB_ID_COL]
    .nunique()
    .reset_index(name="UCDB_Cities")
)

In [14]:
cityseg_size_global = (
    cityseg_city_pop
    .groupby("CitySizeClass")[CITYSEG_ID_COL]
    .nunique()
    .reset_index(name="CitySegments_Cities")
)

In [15]:
size_comparison_global = pd.merge(
    ucdb_size_global,
    cityseg_size_global,
    on="CitySizeClass",
    how="outer"
)

size_comparison_global = size_comparison_global.fillna(0)

size_comparison_global["Coverage_%"] = (
    size_comparison_global["CitySegments_Cities"] /
    size_comparison_global["UCDB_Cities"].replace(0, pd.NA)
) * 100

# Optional: order logically
size_order = [
    "Small (<500k)",
    "Medium (500k–1M)",
    "Large (1–5M)",
    "Very large (5–10M)",
    "Megacity (>10M)"
]

size_comparison_global["CitySizeClass"] = pd.Categorical(
    size_comparison_global["CitySizeClass"],
    categories=size_order,
    ordered=True
)

size_comparison_global = size_comparison_global.sort_values("CitySizeClass")

size_comparison_global

,CitySizeClass,UCDB_Cities,CitySegments_Cities,Coverage_%
3,Small (<500k),12079,4505,37.296134
1,Medium (500k–1M),541,356,65.804067
0,Large (1–5M),432,287,66.435185
4,Very large (5–10M),47,34,72.340426
2,Megacity (>10M),36,32,88.888889
